# Notebook 05: Clinical Complication Extraction with medSpaCy

**Author:** Anthony Amit Biswas

## What this notebook does

Builds a rule-based medSpaCy pipeline for complication detection, contextual classification, validation, and cohort-level extraction.


# **Setup and data validation**

**Installing dependencies**

In [ ]:
# 1. Install Dependencies


!pip install -q \
    "spacy==3.8.2" \
    "medspacy==1.3.1" \
    "pandas>=2.0,<3.0" \
    "tqdm>=4.66,<5.0"

**Imports and logging control**

In [ ]:
# 2. Importing Libraries and Disable Excessive Logging

import gc
import json
import math
import re
import sys
import time
import warnings

from collections import Counter
from pathlib import Path

import medspacy
import pandas as pd
import spacy

from loguru import logger
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# Removing Loguru handlers before any medSpaCy processing.
# This prevents PyRuSH or other dependencies from flooding Colab output.
logger.remove()

print("-" * 75)
print("ENVIRONMENT")
print("-" * 75)

print(f"Python   : {sys.version.split()[0]}")
print(f"pandas   : {pd.__version__}")
print(f"spaCy    : {spacy.__version__}")
print(f"medSpaCy : {medspacy.__version__}")

assert spacy.__version__ == "3.8.2"
assert medspacy.__version__ == "1.3.1"

print("\nEnvironment validated.")

**Mounting Drive and configuring paths**

In [ ]:
# 3. Mounting the Google Drive and Configuring Paths

from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIRECTORY = Path(
    "/content/drive/MyDrive/Dissertation"
)

OUTPUT_DIRECTORY = (
    PROJECT_DIRECTORY
    / "outputs"
)

INPUT_FILE = (
    OUTPUT_DIRECTORY
    / "icu_discharge_summaries_preprocessed.csv.gz"
)

MEDSPACY_OUTPUT_DIRECTORY = (
    OUTPUT_DIRECTORY
    / "medspacy"
)

SHARD_DIRECTORY = (
    MEDSPACY_OUTPUT_DIRECTORY
    / "entity_shards_lexicon_v2"
)

SUMMARY_DIRECTORY = (
    MEDSPACY_OUTPUT_DIRECTORY
    / "summary_outputs"
)

BENCHMARK_DIRECTORY = (
    MEDSPACY_OUTPUT_DIRECTORY
    / "benchmark_outputs"
)

for directory in [
    MEDSPACY_OUTPUT_DIRECTORY,
    SHARD_DIRECTORY,
    SUMMARY_DIRECTORY,
    BENCHMARK_DIRECTORY
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

assert INPUT_FILE.exists(), (
    f"Input file was not found: {INPUT_FILE}"
)

print("-" * 75)
print("PROJECT PATHS")
print("-" * 75)

print(f"Input file          : {INPUT_FILE}")
print(f"medSpaCy outputs    : {MEDSPACY_OUTPUT_DIRECTORY}")
print(f"Entity shards       : {SHARD_DIRECTORY}")
print(f"Summary outputs     : {SUMMARY_DIRECTORY}")
print(f"Benchmark outputs   : {BENCHMARK_DIRECTORY}")

**Validating the complete cohort**

In [ ]:
# 4. Validating Input Cohort

required_columns = {
    "note_id",
    "subject_id",
    "hadm_id",
    "clean_text"
}

cohort_header = pd.read_csv(
    INPUT_FILE,
    nrows=5
)

missing_columns = required_columns.difference(
    cohort_header.columns
)

assert not missing_columns, (
    f"Missing required columns: {sorted(missing_columns)}"
)

total_notes = 0
missing_clean_text = 0
blank_clean_text = 0
character_counts = []

for input_chunk in pd.read_csv(
    INPUT_FILE,
    usecols=list(required_columns),
    chunksize=1000
):
    total_notes += len(input_chunk)

    missing_clean_text += int(
        input_chunk["clean_text"].isna().sum()
    )

    text_series = (
        input_chunk["clean_text"]
        .fillna("")
        .astype(str)
    )

    blank_clean_text += int(
        text_series.str.strip().eq("").sum()
    )

    character_counts.extend(
        text_series.str.len().tolist()
    )

character_statistics = pd.Series(
    character_counts,
    dtype="int64"
).describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

print("-" * 75)
print("INPUT COHORT VALIDATION")
print("-" * 75)

print(f"Total notes        : {total_notes:,}")
print(f"Missing clean_text : {missing_clean_text:,}")
print(f"Blank clean_text   : {blank_clean_text:,}")

print("\nCharacter-count statistics:")
display(
    character_statistics.to_frame(
        name="character_count"
    )
)

assert total_notes == 65_323, (
    f"Expected 65,323 notes but found {total_notes:,}."
)

# **Complication terminology**

**Defining taxonomy**

In [ ]:
# 5. Define Clinical Complication Taxonomy

COMPLICATION_TERMS = {
    "ACUTE_KIDNEY_INJURY": [
        "acute kidney injury",
        "acute renal failure",
        "acute kidney failure",
        "aki",
        "acute on chronic kidney injury",
        "acute on chronic kidney disease"
    ],

    "SEPSIS": [
        "sepsis",
        "severe sepsis",
        "urosepsis"
    ],

    "SEPTIC_SHOCK": [
        "septic shock"
    ],

    "PNEUMONIA": [
        "pneumonia",
        "hospital acquired pneumonia",
        "hospital-acquired pneumonia",
        "ventilator associated pneumonia",
        "ventilator-associated pneumonia",
        "vap"
    ],

    "RESPIRATORY_FAILURE": [
        "respiratory failure",
        "acute respiratory failure",
        "acute on chronic respiratory failure",
        "hypoxemic respiratory failure",
        "hypoxic respiratory failure",
        "hypercapnic respiratory failure",
        "acute hypoxemic respiratory failure",
        "acute hypoxic respiratory failure",
        "acute hypercapnic respiratory failure"
    ],

    "ARDS": [
        "acute respiratory distress syndrome",
        "ards"
    ],

    "ASPIRATION": [
        "aspiration",
        "silent aspiration",
        "aspiration pneumonia",
        "aspiration pneumonitis",
        "pulmonary aspiration"
    ],

    "PNEUMOTHORAX": [
        "pneumothorax",
        "tension pneumothorax"
    ],

    "PLEURAL_EFFUSION": [
        "pleural effusion",
        "pleural effusions"
    ],

    "DEEP_VEIN_THROMBOSIS": [
        "deep vein thrombosis",
        "deep venous thrombosis",
        "dvt"
    ],

    "PULMONARY_EMBOLISM": [
        "pulmonary embolism",
        "pulmonary embolus",
        "pulmonary emboli"
    ],

    "MYOCARDIAL_INFARCTION": [
        "myocardial infarction",
        "acute myocardial infarction",
        "nstemi",
        "stemi",
        "non-st elevation myocardial infarction",
        "non-st-elevation myocardial infarction",
        "st elevation myocardial infarction",
        "st-elevation myocardial infarction"
    ],

    "CARDIAC_ARREST": [
        "cardiac arrest",
        "cardiopulmonary arrest",
        "pea arrest",
        "pulseless electrical activity arrest"
    ],

    "ARRHYTHMIA": [
        "atrial fibrillation",
        "atrial flutter",
        "ventricular tachycardia",
        "ventricular fibrillation",
        "heart block"
    ],

    "STROKE": [
        "ischemic stroke",
        "ischaemic stroke",
        "hemorrhagic stroke",
        "haemorrhagic stroke",
        "cerebral infarction",
        "cva",
        "cerebrovascular accident"
    ],

    "DELIRIUM": [
        "delirium",
        "acute delirium"
    ],

    "ENCEPHALOPATHY": [
        "encephalopathy",
        "acute encephalopathy",
        "metabolic encephalopathy",
        "hepatic encephalopathy",
        "toxic metabolic encephalopathy",
        "toxic-metabolic encephalopathy"
    ],

    "GASTROINTESTINAL_BLEEDING": [
        "gastrointestinal bleeding",
        "gastrointestinal bleed",
        "upper gastrointestinal bleeding",
        "upper gastrointestinal bleed",
        "lower gastrointestinal bleeding",
        "lower gastrointestinal bleed",
        "gi bleeding",
        "gi bleed",
        "hematemesis",
        "haematemesis",
        "melena",
        "melaena",
        "hematochezia",
        "haematochezia",
        "coffee-ground emesis",
        "coffee ground emesis"
    ],

    "CLOSTRIDIOIDES_DIFFICILE_INFECTION": [
        "clostridioides difficile infection",
        "clostridium difficile infection",
        "c difficile infection",
        "c. difficile infection",
        "c diff infection",
        "c. diff infection",
        "c difficile colitis",
        "c. difficile colitis",
        "c diff colitis",
        "c. diff colitis",
        "c.diff colitis"
    ],

    "URINARY_TRACT_INFECTION": [
        "urinary tract infection",
        "catheter associated urinary tract infection",
        "catheter-associated urinary tract infection",
        "uti"
    ],

    "BLOODSTREAM_INFECTION": [
        "bloodstream infection",
        "central line associated bloodstream infection",
        "central line-associated bloodstream infection",
        "clabsi",
        "line infection",
        "bacteremia",
        "bacteraemia",
        "fungemia",
        "fungaemia"
    ],

    "PRESSURE_INJURY": [
        "pressure injury",
        "pressure ulcer",
        "decubitus ulcer"
    ],

    "WOUND_INFECTION": [
        "wound infection",
        "surgical site infection",
        "postoperative wound infection",
        "post-operative wound infection"
    ]
}

number_of_categories = len(COMPLICATION_TERMS)

number_of_terms = sum(
    len(terms)
    for terms in COMPLICATION_TERMS.values()
)

print("-" * 75)
print("COMPLICATION TAXONOMY")
print("-" * 75)

print(f"Categories       : {number_of_categories:,}")
print(f"Lexical variants : {number_of_terms:,}")

**Validating terminology and exporting it**

In [ ]:
# 6. Validating and Exporting Terminology

rule_records = []

for category, terms in COMPLICATION_TERMS.items():
    for term in terms:
        normalized_term = term.lower().strip()

        rule_records.append(
            {
                "category": category,
                "term": normalized_term
            }
        )

complication_rules_df = pd.DataFrame(
    rule_records
)

duplicate_rules = complication_rules_df[
    complication_rules_df.duplicated(
        subset=["term"],
        keep=False
    )
].sort_values("term")

assert duplicate_rules.empty, (
    "Some terms are assigned more than once:\n"
    f"{duplicate_rules}"
)

complication_rules_df = (
    complication_rules_df
    .sort_values(["category", "term"])
    .reset_index(drop=True)
)

RULE_FILE = (
    MEDSPACY_OUTPUT_DIRECTORY
    / "medspacy_complication_rules.csv"
)

complication_rules_df.to_csv(
    RULE_FILE,
    index=False
)

print(f"Rules exported: {len(complication_rules_df):,}")
print(f"Output file   : {RULE_FILE}")

display(
    complication_rules_df.head(25)
)

# **Lightweight medSpaCy pipeline**

**Building a minimal pipeline for faster execution**

In [ ]:
# 7. Build Lightweight medSpaCy Pipeline

from medspacy.ner import TargetRule

# Starting with a blank English spaCy pipeline.
medspacy_nlp = spacy.blank("en")

# Using spaCy's lightweight rule-based sentence segmenter.
medspacy_nlp.add_pipe(
    "sentencizer",
    first=True
)

# Adding only the required medSpaCy components.
medspacy_nlp.add_pipe(
    "medspacy_target_matcher",
    last=True
)

medspacy_nlp.add_pipe(
    "medspacy_context",
    last=True
)

print("-" * 75)
print("LIGHTWEIGHT MEDSPACY PIPELINE")
print("-" * 75)

print(medspacy_nlp.pipe_names)

expected_components = [
    "sentencizer",
    "medspacy_target_matcher",
    "medspacy_context"
]

assert medspacy_nlp.pipe_names == expected_components

print("\nPipeline configuration validated.")

spaCy documents the `sentencizer` as a simple rule-based component that sets sentence boundaries without loading a parser or statistical model. This is more efficient and faster than `PyRuSH`.

**Registering target rules**

In [ ]:
# 8. Registering Target Rules

target_matcher = medspacy_nlp.get_pipe(
    "medspacy_target_matcher"
)

target_rules = [
    TargetRule(
        literal=row.term,
        category=row.category
    )
    for row in complication_rules_df.itertuples(
        index=False
    )
]

target_matcher.add(target_rules)

print(f"Target rules registered: {len(target_rules):,}")

**Creating candidate prefilter**

The prefilter does not determine final entities. It only identifies segments worth passing to medSpaCy.

In [ ]:
# 9. Compiling Candidate-Term Prefilter

all_terms = sorted(
    complication_rules_df["term"].tolist(),
    key=len,
    reverse=True
)

escaped_terms = [
    re.escape(term)
    for term in all_terms
]

candidate_pattern = re.compile(
    r"(?<!\w)(?:"
    + "|".join(escaped_terms)
    + r")(?!\w)",
    flags=re.IGNORECASE
)

print(f"Prefilter terms compiled: {len(all_terms):,}")

# **Safe text segmentation**

**Segmentation of clinical text**

This implementation prioritises line breaks because discharge summaries are highly sectioned documents.

In [ ]:
# 10. Defining Clinical Text Segmentation

def split_clinical_text(
    text,
    max_segment_characters=1500
):
    """
    Divide a clinical note into manageable segments.

    Segmentation first uses line and paragraph boundaries. Exceptionally long
    lines are then divided using punctuation boundaries or fixed-size windows.

    Returns
    -------
    list of dict
        Each dictionary contains segment text and its original character offset.
    """

    if pd.isna(text):
        return []

    text = str(text)

    if not text.strip():
        return []

    segments = []

    # Preserving original offsets while separating at newline boundaries.
    line_pattern = re.compile(
        r"[^\n]+(?:\n+|$)"
    )

    for line_match in line_pattern.finditer(text):
        raw_line = line_match.group(0)
        line_start = line_match.start()

        line_text = raw_line.strip()

        if not line_text:
            continue

        leading_whitespace = (
            len(raw_line)
            - len(raw_line.lstrip())
        )

        true_line_start = (
            line_start
            + leading_whitespace
        )

        if len(line_text) <= max_segment_characters:
            segments.append(
                {
                    "segment_text": line_text,
                    "segment_start_char": true_line_start
                }
            )

            continue

        # Dividing unusually long lines at punctuation where possible.
        sub_pattern = re.compile(
            r".{1,"
            + str(max_segment_characters)
            + r"}(?:[.!?](?=\s|$)|$)",
            flags=re.DOTALL
        )

        for sub_match in sub_pattern.finditer(line_text):
            sub_text = sub_match.group(0).strip()

            if not sub_text:
                continue

            sub_leading = (
                len(sub_match.group(0))
                - len(sub_match.group(0).lstrip())
            )

            segments.append(
                {
                    "segment_text": sub_text,
                    "segment_start_char": (
                        true_line_start
                        + sub_match.start()
                        + sub_leading
                    )
                }
            )

    return segments

**Selecting candidate segments with context neighbours**

To avoid losing cues such as section headings or a preceding negation statement, each matching segment is combined with one neighbouring segment on either side

In [ ]:
# 11. Selecting Candidate Segments

def select_candidate_segments(
    text,
    include_neighbour_segments=True
):
    """
    Select text segments containing at least one target term.

    Adjacent segments can be included to preserve nearby contextual information.
    Overlapping candidate windows are merged before medSpaCy processing.
    """

    segments = split_clinical_text(text)

    if not segments:
        return []

    matching_indices = {
        index
        for index, segment in enumerate(segments)
        if candidate_pattern.search(
            segment["segment_text"]
        )
    }

    if not matching_indices:
        return []

    selected_indices = set(
        matching_indices
    )

    if include_neighbour_segments:
        for index in matching_indices:
            if index > 0:
                selected_indices.add(index - 1)

            if index < len(segments) - 1:
                selected_indices.add(index + 1)

    sorted_indices = sorted(
        selected_indices
    )

    # Merging consecutive selected segments.
    index_groups = []
    current_group = [sorted_indices[0]]

    for index in sorted_indices[1:]:
        if index == current_group[-1] + 1:
            current_group.append(index)
        else:
            index_groups.append(current_group)
            current_group = [index]

    index_groups.append(current_group)

    candidate_windows = []

    for group in index_groups:
        first_segment = segments[group[0]]
        last_segment = segments[group[-1]]

        window_start = first_segment[
            "segment_start_char"
        ]

        last_segment_end = (
            last_segment["segment_start_char"]
            + len(last_segment["segment_text"])
        )

        window_text = str(text)[
            window_start:last_segment_end
        ]

        candidate_windows.append(
            {
                "window_text": window_text,
                "window_start_char": window_start,
                "contains_direct_match": True
            }
        )

    return candidate_windows

# **Extraction**

**Context helper**

In [ ]:
# 12. Defining Context Attribute Helper

def get_context_attributes(entity):
    """
    Safely retrieve medSpaCy ConText attributes.
    """

    return {
        "is_negated": bool(
            getattr(
                entity._,
                "is_negated",
                False
            )
        ),

        "is_uncertain": bool(
            getattr(
                entity._,
                "is_uncertain",
                False
            )
        ),

        "is_historical": bool(
            getattr(
                entity._,
                "is_historical",
                False
            )
        ),

        "is_hypothetical": bool(
            getattr(
                entity._,
                "is_hypothetical",
                False
            )
        ),

        "is_family": bool(
            getattr(
                entity._,
                "is_family",
                False
            )
        )
    }

**Extraction function**

In [ ]:
# 13. Defining medSpaCy Extraction Function

def extract_medspacy_complications(
    note_id,
    subject_id,
    hadm_id,
    note_text
):
    """
    Extract candidate complication mentions from one discharge summary.

    Only candidate-containing windows are processed by medSpaCy. Character
    offsets are mapped back to the original discharge summary.
    """

    if pd.isna(note_text):
        return [], {
            "candidate_windows": 0,
            "characters_processed": 0,
            "original_characters": 0
        }

    note_text = str(note_text)

    if not note_text.strip():
        return [], {
            "candidate_windows": 0,
            "characters_processed": 0,
            "original_characters": len(note_text)
        }

    candidate_windows = select_candidate_segments(
        note_text
    )

    if not candidate_windows:
        return [], {
            "candidate_windows": 0,
            "characters_processed": 0,
            "original_characters": len(note_text)
        }

    extracted_records = []
    seen_mentions = set()
    characters_processed = 0

    for window_number, window in enumerate(
        candidate_windows
    ):
        window_text = window["window_text"]
        window_start = window[
            "window_start_char"
        ]

        characters_processed += len(
            window_text
        )

        doc = medspacy_nlp(
            window_text
        )

        for entity in doc.ents:
            original_start = (
                window_start
                + entity.start_char
            )

            original_end = (
                window_start
                + entity.end_char
            )

            duplicate_key = (
                entity.label_,
                original_start,
                original_end
            )

            if duplicate_key in seen_mentions:
                continue

            seen_mentions.add(
                duplicate_key
            )

            context = get_context_attributes(
                entity
            )

            is_current_affirmed = not any(
                context.values()
            )

            sentence_text = ""

            try:
                sentence_text = (
                    entity.sent.text.strip()
                )
            except Exception:
                sentence_text = window_text.strip()

            extracted_records.append(
                {
                    "note_id": note_id,
                    "subject_id": subject_id,
                    "hadm_id": hadm_id,
                    "entity_text": entity.text,
                    "normalized_text": (
                        entity.text
                        .lower()
                        .strip()
                    ),
                    "category": entity.label_,
                    "start_char": original_start,
                    "end_char": original_end,
                    "sentence_text": sentence_text,
                    "window_number": window_number,
                    "is_negated": context[
                        "is_negated"
                    ],
                    "is_uncertain": context[
                        "is_uncertain"
                    ],
                    "is_historical": context[
                        "is_historical"
                    ],
                    "is_hypothetical": context[
                        "is_hypothetical"
                    ],
                    "is_family": context[
                        "is_family"
                    ],
                    "is_current_affirmed": (
                        is_current_affirmed
                    )
                }
            )

    processing_metadata = {
        "candidate_windows": len(
            candidate_windows
        ),
        "characters_processed": (
            characters_processed
        ),
        "original_characters": len(
            note_text
        )
    }

    return (
        extracted_records,
        processing_metadata
    )

# **Controlled tests**

**Synthetic tests**

In [ ]:
# 14. Controlled Synthetic Tests

synthetic_examples = [
    (
        "SYNTHETIC_01",
        "The patient developed acute kidney injury during admission."
    ),
    (
        "SYNTHETIC_02",
        "There was no evidence of pulmonary embolism."
    ),
    (
        "SYNTHETIC_03",
        "Possible aspiration pneumonia was considered."
    ),
    (
        "SYNTHETIC_04",
        "The patient has a history of atrial fibrillation."
    ),
    (
        "SYNTHETIC_05",
        "The admission was complicated by septic shock and acute respiratory failure."
    ),
    (
        "SYNTHETIC_06",
        "The patient was monitored because of the risk of deep vein thrombosis."
    )
]

synthetic_records = []

for example_id, example_text in synthetic_examples:
    mentions, metadata = extract_medspacy_complications(
        note_id=example_id,
        subject_id="SYNTHETIC",
        hadm_id="SYNTHETIC",
        note_text=example_text
    )

    synthetic_records.extend(
        mentions
    )

synthetic_results_df = pd.DataFrame(
    synthetic_records
)

display_columns = [
    "note_id",
    "entity_text",
    "category",
    "is_negated",
    "is_uncertain",
    "is_historical",
    "is_hypothetical",
    "is_family",
    "is_current_affirmed"
]

print("-" * 75)
print("SYNTHETIC TEST RESULTS")
print("-" * 75)

print(f"Mentions detected: {len(synthetic_results_df):,}")

display(
    synthetic_results_df[
        display_columns
    ]
)

**Loading five notes**

In [ ]:
# 15. Loading Five-Note Technical Sample

five_note_sample = pd.read_csv(
    INPUT_FILE,
    usecols=[
        "note_id",
        "subject_id",
        "hadm_id",
        "clean_text"
    ],
    nrows=5
)

five_note_sample["character_count"] = (
    five_note_sample["clean_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

display(
    five_note_sample[
        [
            "note_id",
            "subject_id",
            "hadm_id",
            "character_count"
        ]
    ]
)

**Runing five-note benchmark**

In [ ]:
# 16. Five-Note Performance and Output Test

five_note_records = []
five_note_timing_records = []

overall_start = time.perf_counter()

for note_number, row in enumerate(
    five_note_sample.itertuples(
        index=False
    ),
    start=1
):
    note_start = time.perf_counter()

    mentions, metadata = (
        extract_medspacy_complications(
            note_id=row.note_id,
            subject_id=row.subject_id,
            hadm_id=row.hadm_id,
            note_text=row.clean_text
        )
    )

    elapsed_seconds = (
        time.perf_counter()
        - note_start
    )

    five_note_records.extend(
        mentions
    )

    five_note_timing_records.append(
        {
            "note_id": row.note_id,
            "original_characters": metadata[
                "original_characters"
            ],
            "candidate_windows": metadata[
                "candidate_windows"
            ],
            "characters_processed": metadata[
                "characters_processed"
            ],
            "mentions_extracted": len(
                mentions
            ),
            "processing_seconds": (
                elapsed_seconds
            )
        }
    )

    print(
        f"Note {note_number}/5 completed | "
        f"windows={metadata['candidate_windows']:,} | "
        f"mentions={len(mentions):,} | "
        f"time={elapsed_seconds:.3f}s"
    )

overall_seconds = (
    time.perf_counter()
    - overall_start
)

five_note_results_df = pd.DataFrame(
    five_note_records
)

five_note_timing_df = pd.DataFrame(
    five_note_timing_records
)

print("\n" + "-" * 75)
print("FIVE-NOTE BENCHMARK")
print("-" * 75)

print(f"Total runtime : {overall_seconds:.3f} seconds")
print(
    f"Mean per note : "
    f"{overall_seconds / len(five_note_sample):.3f} seconds"
)

display(
    five_note_timing_df
)

if not five_note_results_df.empty:
    display(
        five_note_results_df[
            [
                "note_id",
                "entity_text",
                "category",
                "is_negated",
                "is_uncertain",
                "is_historical",
                "is_hypothetical",
                "is_current_affirmed",
                "sentence_text"
            ]
        ].head(50)
    )
else:
    print("No target complications were detected.")

**Adding task-specific context detection**

The default medSpaCy rules failed to classify: risk of deep vein thrombosis as hypothetical. We will preserve medSpaCy’s native attributes and add transparent task-specific flags.

In [ ]:
# 17. Defining Task-Specific Context Rules

CUSTOM_UNCERTAINTY_PATTERN = re.compile(
    r"\b(?:"
    r"possible|possibly|probable|probably|suspected|"
    r"suspicious for|concerning for|concern for|"
    r"cannot exclude|could represent|may represent|"
    r"rule out|r/o|evaluate for|evaluation for"
    r")\b",
    flags=re.IGNORECASE
)

CUSTOM_HYPOTHETICAL_PATTERN = re.compile(
    r"\b(?:"
    r"risk of|at risk for|monitor(?:ed|ing)? for|"
    r"watch(?:ed|ing)? for|prophylaxis (?:for|against)|"
    r"prevention of|prevent(?:ion|ing)?|"
    r"screen(?:ed|ing)? for"
    r")\b",
    flags=re.IGNORECASE
)

print("Task-specific context patterns compiled.")

**Defining custom context helper**

In [ ]:
# 18. Defining Task-Specific Context Helper

def get_task_specific_context(entity):
    """
    Detect task-specific uncertainty and hypothetical cues appearing before
    an entity within the same sentence.

    Only the final 120 characters before the entity are examined to reduce
    inappropriate long-distance cue assignment.
    """

    try:
        sentence = entity.sent

        relative_entity_start = (
            entity.start_char
            - sentence.start_char
        )

        prefix_text = sentence.text[
            :relative_entity_start
        ]

    except Exception:
        prefix_text = ""

    local_prefix = prefix_text[-120:]

    custom_uncertain = bool(
        CUSTOM_UNCERTAINTY_PATTERN.search(
            local_prefix
        )
    )

    custom_hypothetical = bool(
        CUSTOM_HYPOTHETICAL_PATTERN.search(
            local_prefix
        )
    )

    return {
        "custom_uncertain": custom_uncertain,
        "custom_hypothetical": custom_hypothetical,
        "context_prefix": local_prefix.strip()
    }

**The extraction function**

In [ ]:
# 19. Defining Extraction Function with Combined Context Handling

def extract_medspacy_complications(
    note_id,
    subject_id,
    hadm_id,
    note_text
):
    """
    Extract candidate clinical complication mentions from one discharge summary.

    medSpaCy native ConText attributes are retained. Task-specific uncertainty
    and hypothetical cues are added separately and combined into final
    operational status variables.
    """

    metadata_template = {
        "candidate_windows": 0,
        "characters_processed": 0,
        "original_characters": 0
    }

    if pd.isna(note_text):
        return [], metadata_template.copy()

    note_text = str(note_text)

    metadata_template[
        "original_characters"
    ] = len(note_text)

    if not note_text.strip():
        return [], metadata_template.copy()

    candidate_windows = select_candidate_segments(
        note_text
    )

    if not candidate_windows:
        return [], metadata_template.copy()

    extracted_records = []
    seen_mentions = set()
    characters_processed = 0

    for window_number, window in enumerate(
        candidate_windows
    ):
        window_text = window["window_text"]
        window_start = window[
            "window_start_char"
        ]

        characters_processed += len(
            window_text
        )

        doc = medspacy_nlp(
            window_text
        )

        for entity in doc.ents:
            original_start = (
                window_start
                + entity.start_char
            )

            original_end = (
                window_start
                + entity.end_char
            )

            duplicate_key = (
                entity.label_,
                original_start,
                original_end
            )

            if duplicate_key in seen_mentions:
                continue

            seen_mentions.add(
                duplicate_key
            )

            native_context = (
                get_context_attributes(
                    entity
                )
            )

            custom_context = (
                get_task_specific_context(
                    entity
                )
            )

            combined_uncertain = bool(
                native_context["is_uncertain"]
                or custom_context[
                    "custom_uncertain"
                ]
            )

            combined_hypothetical = bool(
                native_context["is_hypothetical"]
                or custom_context[
                    "custom_hypothetical"
                ]
            )

            is_current_affirmed = not any(
                [
                    native_context[
                        "is_negated"
                    ],
                    combined_uncertain,
                    native_context[
                        "is_historical"
                    ],
                    combined_hypothetical,
                    native_context[
                        "is_family"
                    ]
                ]
            )

            try:
                sentence_text = (
                    entity.sent.text.strip()
                )
            except Exception:
                sentence_text = (
                    window_text.strip()
                )

            extracted_records.append(
                {
                    "note_id": note_id,
                    "subject_id": subject_id,
                    "hadm_id": hadm_id,

                    "entity_text": entity.text,

                    "normalized_text": (
                        entity.text
                        .lower()
                        .strip()
                    ),

                    "category": entity.label_,

                    "start_char": original_start,
                    "end_char": original_end,

                    "sentence_text": sentence_text,
                    "window_number": window_number,

                    # Native medSpaCy attributes
                    "is_negated": native_context[
                        "is_negated"
                    ],

                    "native_is_uncertain": (
                        native_context[
                            "is_uncertain"
                        ]
                    ),

                    "is_historical": native_context[
                        "is_historical"
                    ],

                    "native_is_hypothetical": (
                        native_context[
                            "is_hypothetical"
                        ]
                    ),

                    "is_family": native_context[
                        "is_family"
                    ],

                    # Task-specific additions
                    "custom_uncertain": (
                        custom_context[
                            "custom_uncertain"
                        ]
                    ),

                    "custom_hypothetical": (
                        custom_context[
                            "custom_hypothetical"
                        ]
                    ),

                    "context_prefix": (
                        custom_context[
                            "context_prefix"
                        ]
                    ),

                    # Combined operational attributes
                    "is_uncertain": (
                        combined_uncertain
                    ),

                    "is_hypothetical": (
                        combined_hypothetical
                    ),

                    "is_current_affirmed": (
                        is_current_affirmed
                    )
                }
            )

    processing_metadata = {
        "candidate_windows": len(
            candidate_windows
        ),

        "characters_processed": (
            characters_processed
        ),

        "original_characters": len(
            note_text
        )
    }

    return (
        extracted_records,
        processing_metadata
    )

**Validating the context logic**

**Expanded synthetic tests**

In [ ]:
# 20. Expanded Context Validation

context_test_examples = [
    (
        "TEST_01",
        "The patient developed acute kidney injury."
    ),
    (
        "TEST_02",
        "There was no evidence of pulmonary embolism."
    ),
    (
        "TEST_03",
        "Possible aspiration pneumonia was considered."
    ),
    (
        "TEST_04",
        "The patient has a history of atrial fibrillation."
    ),
    (
        "TEST_05",
        "The patient was monitored because of the risk of deep vein thrombosis."
    ),
    (
        "TEST_06",
        "The team planned to rule out pneumonia."
    ),
    (
        "TEST_07",
        "Findings were concerning for pneumonia."
    ),
    (
        "TEST_08",
        "The admission was complicated by septic shock."
    ),
    (
        "TEST_09",
        "DVT prophylaxis was continued."
    )
]

context_test_records = []

for example_id, example_text in context_test_examples:
    mentions, _ = extract_medspacy_complications(
        note_id=example_id,
        subject_id="SYNTHETIC",
        hadm_id="SYNTHETIC",
        note_text=example_text
    )

    context_test_records.extend(
        mentions
    )

context_test_df = pd.DataFrame(
    context_test_records
)

context_test_columns = [
    "note_id",
    "entity_text",
    "is_negated",
    "native_is_uncertain",
    "custom_uncertain",
    "is_uncertain",
    "is_historical",
    "native_is_hypothetical",
    "custom_hypothetical",
    "is_hypothetical",
    "is_current_affirmed"
]

print("-" * 75)
print("EXPANDED CONTEXT VALIDATION")
print("-" * 75)

display(
    context_test_df[
        context_test_columns
    ]
)

# **100-note benchmark**

**Loading a reproducible 100-note benchmark**

Using a fixed random seed rather than simply selecting the first 100 records.

In [ ]:
# 21. Create Reproducible 100-Note Benchmark

BENCHMARK_SIZE = 100
RANDOM_SEED = 42

benchmark_source_df = pd.read_csv(
    INPUT_FILE,
    usecols=[
        "note_id",
        "subject_id",
        "hadm_id",
        "clean_text"
    ]
)

assert len(benchmark_source_df) == 65_323

benchmark_100_df = (
    benchmark_source_df
    .sample(
        n=BENCHMARK_SIZE,
        random_state=RANDOM_SEED
    )
    .sort_index()
    .reset_index(drop=True)
)

BENCHMARK_SAMPLE_FILE = (
    BENCHMARK_DIRECTORY
    / "medspacy_benchmark_100_notes.csv.gz"
)

benchmark_100_df.to_csv(
    BENCHMARK_SAMPLE_FILE,
    index=False,
    compression="gzip"
)

print("-" * 75)
print("100-NOTE BENCHMARK SAMPLE")
print("-" * 75)

print(f"Notes selected : {len(benchmark_100_df):,}")
print(f"Random seed    : {RANDOM_SEED}")
print(f"Output file    : {BENCHMARK_SAMPLE_FILE}")

display(
    benchmark_100_df[
        [
            "note_id",
            "subject_id",
            "hadm_id"
        ]
    ].head()
)

In [ ]:
del benchmark_source_df
gc.collect()

print("Full benchmark source dataframe released from memory.")

**Running the 100-note benchmark**

In [ ]:
# 22. Running 100-Note Benchmark

benchmark_mention_records = []
benchmark_timing_records = []

benchmark_start_time = time.perf_counter()

for note_number, row in enumerate(
    benchmark_100_df.itertuples(
        index=False
    ),
    start=1
):
    note_start_time = time.perf_counter()

    mentions, metadata = (
        extract_medspacy_complications(
            note_id=row.note_id,
            subject_id=row.subject_id,
            hadm_id=row.hadm_id,
            note_text=row.clean_text
        )
    )

    note_elapsed_seconds = (
        time.perf_counter()
        - note_start_time
    )

    benchmark_mention_records.extend(
        mentions
    )

    benchmark_timing_records.append(
        {
            "note_id": row.note_id,

            "original_characters": (
                metadata[
                    "original_characters"
                ]
            ),

            "candidate_windows": (
                metadata[
                    "candidate_windows"
                ]
            ),

            "characters_processed": (
                metadata[
                    "characters_processed"
                ]
            ),

            "mentions_extracted": len(
                mentions
            ),

            "processing_seconds": (
                note_elapsed_seconds
            )
        }
    )

    if (
        note_number == 1
        or note_number % 10 == 0
        or note_number == BENCHMARK_SIZE
    ):
        print(
            f"Processed {note_number:>3}/"
            f"{BENCHMARK_SIZE} notes"
        )

benchmark_elapsed_seconds = (
    time.perf_counter()
    - benchmark_start_time
)

benchmark_mentions_df = pd.DataFrame(
    benchmark_mention_records
)

benchmark_timing_df = pd.DataFrame(
    benchmark_timing_records
)

print("\n" + "-" * 75)
print("100-NOTE BENCHMARK RESULTS")
print("-" * 75)

print(
    f"Notes processed       : "
    f"{len(benchmark_100_df):,}"
)

print(
    f"Mentions extracted    : "
    f"{len(benchmark_mentions_df):,}"
)

print(
    f"Total runtime         : "
    f"{benchmark_elapsed_seconds:.3f} seconds"
)

print(
    f"Mean runtime per note : "
    f"{benchmark_elapsed_seconds / BENCHMARK_SIZE:.4f} seconds"
)

print(
    f"Notes per second      : "
    f"{BENCHMARK_SIZE / benchmark_elapsed_seconds:.2f}"
)

display(
    benchmark_timing_df[
        [
            "original_characters",
            "candidate_windows",
            "characters_processed",
            "mentions_extracted",
            "processing_seconds"
        ]
    ].describe()
)

**Estimating full-cohort runtime**

In [ ]:
# 23. Estimating Full-Cohort Runtime

estimated_full_seconds = (
    benchmark_elapsed_seconds
    / BENCHMARK_SIZE
    * total_notes
)

estimated_full_minutes = (
    estimated_full_seconds
    / 60
)

estimated_full_hours = (
    estimated_full_minutes
    / 60
)

print("-" * 75)
print("ESTIMATED FULL-COHORT RUNTIME")
print("-" * 75)

print(
    f"Estimated seconds : "
    f"{estimated_full_seconds:,.1f}"
)

print(
    f"Estimated minutes : "
    f"{estimated_full_minutes:,.1f}"
)

print(
    f"Estimated hours   : "
    f"{estimated_full_hours:,.2f}"
)

**Saving the benchmark outputs**

In [ ]:
# 24. Exporting Benchmark Results

BENCHMARK_MENTIONS_FILE = (
    BENCHMARK_DIRECTORY
    / "medspacy_benchmark_100_mentions.csv.gz"
)

BENCHMARK_TIMING_FILE = (
    BENCHMARK_DIRECTORY
    / "medspacy_benchmark_100_timing.csv"
)

BENCHMARK_STATISTICS_FILE = (
    BENCHMARK_DIRECTORY
    / "medspacy_benchmark_statistics.csv"
)

benchmark_mentions_df.to_csv(
    BENCHMARK_MENTIONS_FILE,
    index=False,
    compression="gzip"
)

benchmark_timing_df.to_csv(
    BENCHMARK_TIMING_FILE,
    index=False
)

benchmark_statistics_df = pd.DataFrame(
    [
        {
            "benchmark_notes": (
                BENCHMARK_SIZE
            ),

            "mentions_extracted": (
                len(
                    benchmark_mentions_df
                )
            ),

            "runtime_seconds": (
                benchmark_elapsed_seconds
            ),

            "mean_seconds_per_note": (
                benchmark_elapsed_seconds
                / BENCHMARK_SIZE
            ),

            "notes_per_second": (
                BENCHMARK_SIZE
                / benchmark_elapsed_seconds
            ),

            "estimated_full_seconds": (
                estimated_full_seconds
            ),

            "estimated_full_minutes": (
                estimated_full_minutes
            ),

            "estimated_full_hours": (
                estimated_full_hours
            )
        }
    ]
)

benchmark_statistics_df.to_csv(
    BENCHMARK_STATISTICS_FILE,
    index=False
)

print("Benchmark outputs saved successfully.")

**Benchmark context distribution**

In [ ]:
# 25. Inspecting Benchmark Context Distribution

if benchmark_mentions_df.empty:
    print(
        "No mentions were extracted from "
        "the benchmark sample."
    )

else:
    benchmark_context_summary = pd.DataFrame(
        {
            "context_status": [
                "negated",
                "uncertain",
                "historical",
                "hypothetical",
                "family",
                "current_affirmed"
            ],

            "mention_count": [
                int(
                    benchmark_mentions_df[
                        "is_negated"
                    ].sum()
                ),

                int(
                    benchmark_mentions_df[
                        "is_uncertain"
                    ].sum()
                ),

                int(
                    benchmark_mentions_df[
                        "is_historical"
                    ].sum()
                ),

                int(
                    benchmark_mentions_df[
                        "is_hypothetical"
                    ].sum()
                ),

                int(
                    benchmark_mentions_df[
                        "is_family"
                    ].sum()
                ),

                int(
                    benchmark_mentions_df[
                        "is_current_affirmed"
                    ].sum()
                )
            ]
        }
    )

    benchmark_context_summary[
        "percentage_of_mentions"
    ] = (
        benchmark_context_summary[
            "mention_count"
        ]
        / len(benchmark_mentions_df)
        * 100
    )

    display(
        benchmark_context_summary
    )

# **Full resumable extraction**

**Full-run configuration**

The execution flag is deliberately set to `False.`

In [ ]:
# 26. Configuring Full-Cohort Extraction

FULL_RUN_CHUNK_SIZE = 250

FULL_RUN_COLUMNS = [
    "note_id",
    "subject_id",
    "hadm_id",
    "clean_text"
]

MENTION_OUTPUT_COLUMNS = [
    "note_id",
    "subject_id",
    "hadm_id",
    "entity_text",
    "normalized_text",
    "category",
    "start_char",
    "end_char",
    "sentence_text",
    "window_number",
    "is_negated",
    "native_is_uncertain",
    "is_historical",
    "native_is_hypothetical",
    "is_family",
    "custom_uncertain",
    "custom_hypothetical",
    "context_prefix",
    "is_uncertain",
    "is_hypothetical",
    "is_current_affirmed"
]

MANIFEST_FILE = (
    MEDSPACY_OUTPUT_DIRECTORY
    / "medspacy_processing_manifest.csv"
)

RUN_FULL_EXTRACTION = False

expected_number_of_shards = math.ceil(
    total_notes
    / FULL_RUN_CHUNK_SIZE
)

print("-" * 75)
print("FULL EXTRACTION CONFIGURATION")
print("-" * 75)

print(f"Total notes            : {total_notes:,}")
print(f"Notes per shard        : {FULL_RUN_CHUNK_SIZE:,}")
print(f"Expected shards        : {expected_number_of_shards:,}")
print(f"Run enabled            : {RUN_FULL_EXTRACTION}")
print(f"Manifest               : {MANIFEST_FILE}")

**Inspecting existing medSpaCy shards**

In [ ]:
# 27. Inspecting Existing Full-Run Shards

existing_full_shards = sorted(
    SHARD_DIRECTORY.glob(
        "medspacy_mentions_shard_*.csv.gz"
    )
)

print("-" * 75)
print("EXISTING SHARD INSPECTION")
print("-" * 75)

print(
    f"Matching full-run shards found: "
    f"{len(existing_full_shards):,}"
)

for shard_file in existing_full_shards[:10]:
    print(shard_file.name)

if len(existing_full_shards) > 10:
    print("...")

**Full extraction function**

In [ ]:
# 28. Defining Resumable Full Extraction


def run_full_medspacy_extraction():
    """
    Process the complete preprocessed cohort in resumable shards.

    Existing shard files are skipped. One manifest row is written per shard.
    Empty mention shards are also saved with the correct schema.
    """

    manifest_records = []

    full_start_time = time.perf_counter()

    input_reader = pd.read_csv(
        INPUT_FILE,
        usecols=FULL_RUN_COLUMNS,
        chunksize=FULL_RUN_CHUNK_SIZE
    )

    for shard_index, input_chunk in enumerate(
        input_reader
    ):
        shard_number = shard_index + 1

        shard_file = (
            SHARD_DIRECTORY
            / (
                "medspacy_mentions_shard_"
                f"{shard_number:04d}.csv.gz"
            )
        )

        if shard_file.exists():
            print(
                f"Shard {shard_number:04d}/"
                f"{expected_number_of_shards:04d} "
                f"already exists — skipped."
            )

            continue

        shard_start_time = time.perf_counter()

        shard_mention_records = []

        total_candidate_windows = 0
        total_original_characters = 0
        total_characters_processed = 0

        notes_with_mentions = 0
        notes_without_mentions = 0

        for row in input_chunk.itertuples(
            index=False
        ):
            mentions, metadata = (
                extract_medspacy_complications(
                    note_id=row.note_id,
                    subject_id=row.subject_id,
                    hadm_id=row.hadm_id,
                    note_text=row.clean_text
                )
            )

            shard_mention_records.extend(
                mentions
            )

            total_candidate_windows += (
                metadata[
                    "candidate_windows"
                ]
            )

            total_original_characters += (
                metadata[
                    "original_characters"
                ]
            )

            total_characters_processed += (
                metadata[
                    "characters_processed"
                ]
            )

            if mentions:
                notes_with_mentions += 1
            else:
                notes_without_mentions += 1

        shard_mentions_df = pd.DataFrame(
            shard_mention_records,
            columns=MENTION_OUTPUT_COLUMNS
        )

        shard_mentions_df.to_csv(
            shard_file,
            index=False,
            compression="gzip"
        )

        shard_elapsed_seconds = (
            time.perf_counter()
            - shard_start_time
        )

        first_input_row = (
            shard_index
            * FULL_RUN_CHUNK_SIZE
        )

        last_input_row = (
            first_input_row
            + len(input_chunk)
            - 1
        )

        manifest_record = {
            "shard_number": shard_number,
            "shard_file": shard_file.name,

            "first_input_row": (
                first_input_row
            ),

            "last_input_row": (
                last_input_row
            ),

            "first_note_id": (
                input_chunk.iloc[0][
                    "note_id"
                ]
            ),

            "last_note_id": (
                input_chunk.iloc[-1][
                    "note_id"
                ]
            ),

            "notes_processed": len(
                input_chunk
            ),

            "notes_with_mentions": (
                notes_with_mentions
            ),

            "notes_without_mentions": (
                notes_without_mentions
            ),

            "candidate_windows": (
                total_candidate_windows
            ),

            "original_characters": (
                total_original_characters
            ),

            "characters_processed": (
                total_characters_processed
            ),

            "mentions_extracted": len(
                shard_mentions_df
            ),

            "processing_seconds": (
                shard_elapsed_seconds
            )
        }

        manifest_records.append(
            manifest_record
        )

        current_manifest_df = pd.DataFrame(
            manifest_records
        )

        if MANIFEST_FILE.exists():
            previous_manifest_df = pd.read_csv(
                MANIFEST_FILE
            )

            current_manifest_df = pd.concat(
                [
                    previous_manifest_df,
                    current_manifest_df
                ],
                ignore_index=True
            )

        current_manifest_df = (
            current_manifest_df
            .drop_duplicates(
                subset=["shard_number"],
                keep="last"
            )
            .sort_values("shard_number")
            .reset_index(drop=True)
        )

        current_manifest_df.to_csv(
            MANIFEST_FILE,
            index=False
        )

        # Preventing manifest rows from being duplicated on
        # subsequent iterations.
        manifest_records = []

        print(
            f"Shard {shard_number:04d}/"
            f"{expected_number_of_shards:04d} completed | "
            f"notes={len(input_chunk):,} | "
            f"mentions={len(shard_mentions_df):,} | "
            f"time={shard_elapsed_seconds:.2f}s"
        )

        del shard_mentions_df
        del shard_mention_records

        gc.collect()

    total_elapsed_seconds = (
        time.perf_counter()
        - full_start_time
    )

    print("\n" + "-" * 75)
    print("FULL EXTRACTION CALL COMPLETED")
    print("-" * 75)

    print(
        f"Elapsed time in this session: "
        f"{total_elapsed_seconds / 60:.2f} minutes"
    )

In [ ]:
# 29. Executing Full Extraction


if RUN_FULL_EXTRACTION:
    run_full_medspacy_extraction()

else:
    print(
        "Full extraction is disabled.\n"
        "Set RUN_FULL_EXTRACTION = True "
        "only after reviewing the benchmark."
    )

# **Production validation**

**Validating shard presence**

In [ ]:
# 30. Validate Production Shard Count


production_shards = sorted(
    SHARD_DIRECTORY.glob(
        "medspacy_mentions_shard_*.csv.gz"
    )
)

print("-" * 75)
print("PRODUCTION SHARD VALIDATION")
print("-" * 75)

print(
    f"Expected shard files : "
    f"{expected_number_of_shards:,}"
)

print(
    f"Found shard files    : "
    f"{len(production_shards):,}"
)

assert (
    len(production_shards)
    == expected_number_of_shards
), (
    "The number of production shards "
    "does not match the expected count."
)

print("Production shard count validated.")

**Validating shard numbering**

In [ ]:
# 31. Validate Shard Numbering

found_shard_numbers = []

for shard_file in production_shards:
    match = re.search(
        r"medspacy_mentions_shard_(\d{4})\.csv\.gz$",
        shard_file.name
    )

    assert match is not None, (
        f"Unexpected shard filename: "
        f"{shard_file.name}"
    )

    found_shard_numbers.append(
        int(match.group(1))
    )

expected_shard_numbers = list(
    range(
        1,
        expected_number_of_shards + 1
    )
)

missing_shard_numbers = sorted(
    set(expected_shard_numbers)
    - set(found_shard_numbers)
)

duplicate_shard_numbers = sorted(
    number
    for number, count in Counter(
        found_shard_numbers
    ).items()
    if count > 1
)

print(
    f"Missing shard numbers   : "
    f"{missing_shard_numbers}"
)

print(
    f"Duplicate shard numbers : "
    f"{duplicate_shard_numbers}"
)

assert not missing_shard_numbers
assert not duplicate_shard_numbers

print("Shard numbering validated.")

**Validating manifest coverage**

In [ ]:
# 32. Validate Processing Manifest

assert MANIFEST_FILE.exists(), (
    "Processing manifest was not found."
)

manifest_df = pd.read_csv(
    MANIFEST_FILE
)

manifest_df = (
    manifest_df
    .drop_duplicates(
        subset=["shard_number"],
        keep="last"
    )
    .sort_values("shard_number")
    .reset_index(drop=True)
)

print("-" * 75)
print("PROCESSING MANIFEST VALIDATION")
print("-" * 75)

print(
    f"Manifest rows       : "
    f"{len(manifest_df):,}"
)

print(
    f"Notes processed     : "
    f"{manifest_df['notes_processed'].sum():,}"
)

print(
    f"Mentions extracted  : "
    f"{manifest_df['mentions_extracted'].sum():,}"
)

print(
    f"Notes with mentions : "
    f"{manifest_df['notes_with_mentions'].sum():,}"
)

print(
    f"Notes without       : "
    f"{manifest_df['notes_without_mentions'].sum():,}"
)

assert len(manifest_df) == (
    expected_number_of_shards
)

assert int(
    manifest_df[
        "notes_processed"
    ].sum()
) == total_notes

assert int(
    manifest_df[
        "notes_with_mentions"
    ].sum()
    + manifest_df[
        "notes_without_mentions"
    ].sum()
) == total_notes

assert manifest_df[
    "first_input_row"
].iloc[0] == 0

assert manifest_df[
    "last_input_row"
].iloc[-1] == total_notes - 1

expected_next_rows = (
    manifest_df[
        "last_input_row"
    ].iloc[:-1].to_numpy()
    + 1
)

actual_next_rows = (
    manifest_df[
        "first_input_row"
    ].iloc[1:].to_numpy()
)

assert (
    expected_next_rows
    == actual_next_rows
).all()

manifest_df.to_csv(
    MANIFEST_FILE,
    index=False
)

print("\nManifest coverage validated.")

**Validating each shard schema and row count**

In [ ]:
# 33. Validate Shard Schemas and Mention Counts

validated_mentions = 0
schema_errors = []

expected_schema = set(
    MENTION_OUTPUT_COLUMNS
)

for shard_file in tqdm(
    production_shards,
    desc="Validating shards"
):
    shard_df = pd.read_csv(
        shard_file
    )

    found_schema = set(
        shard_df.columns
    )

    if found_schema != expected_schema:
        schema_errors.append(
            {
                "file": shard_file.name,
                "missing_columns": sorted(
                    expected_schema
                    - found_schema
                ),
                "unexpected_columns": sorted(
                    found_schema
                    - expected_schema
                )
            }
        )

    validated_mentions += len(
        shard_df
    )

print(
    f"Mentions counted from shards: "
    f"{validated_mentions:,}"
)

print(
    f"Mentions recorded in manifest: "
    f"{manifest_df['mentions_extracted'].sum():,}"
)

assert not schema_errors, schema_errors

assert validated_mentions == int(
    manifest_df[
        "mentions_extracted"
    ].sum()
)

print("Shard schemas and mention counts validated.")

# **Streaming summaries**

**Accumulating mention statistics**

In [ ]:
# 34. Stream Production Mentions and Accumulate Statistics


category_counter = Counter()
normalized_mention_counter = Counter()

context_counter = Counter()

notes_with_any_mention = set()
notes_with_affirmed_mention = set()

total_mentions_scanned = 0
missing_normalized_mentions = 0
blank_normalized_mentions = 0

for shard_file in tqdm(
    production_shards,
    desc="Scanning mention shards"
):
    for mention_chunk in pd.read_csv(
        shard_file,
        chunksize=100_000
    ):
        total_mentions_scanned += len(
            mention_chunk
        )

        category_counter.update(
            mention_chunk[
                "category"
            ]
            .dropna()
            .astype(str)
        )

        normalized_series = (
            mention_chunk[
                "normalized_text"
            ]
        )

        missing_normalized_mentions += int(
            normalized_series.isna().sum()
        )

        normalized_clean = (
            normalized_series
            .dropna()
            .astype(str)
            .str.strip()
        )

        blank_normalized_mentions += int(
            normalized_clean.eq("").sum()
        )

        normalized_clean = normalized_clean[
            normalized_clean.ne("")
        ]

        normalized_mention_counter.update(
            normalized_clean
        )

        for context_column in [
            "is_negated",
            "is_uncertain",
            "is_historical",
            "is_hypothetical",
            "is_family",
            "is_current_affirmed",
            "custom_uncertain",
            "custom_hypothetical"
        ]:
            context_counter[
                context_column
            ] += int(
                mention_chunk[
                    context_column
                ]
                .fillna(False)
                .astype(bool)
                .sum()
            )

        notes_with_any_mention.update(
            mention_chunk[
                "note_id"
            ]
            .dropna()
            .astype(str)
        )

        affirmed_chunk = mention_chunk[
            mention_chunk[
                "is_current_affirmed"
            ]
            .fillna(False)
            .astype(bool)
        ]

        notes_with_affirmed_mention.update(
            affirmed_chunk[
                "note_id"
            ]
            .dropna()
            .astype(str)
        )

print("-" * 75)
print("STREAMING STATISTICS")
print("-" * 75)

print(
    f"Mentions scanned            : "
    f"{total_mentions_scanned:,}"
)

print(
    f"Missing normalized mentions : "
    f"{missing_normalized_mentions:,}"
)

print(
    f"Blank normalized mentions   : "
    f"{blank_normalized_mentions:,}"
)

print(
    f"Unique normalized mentions  : "
    f"{len(normalized_mention_counter):,}"
)

print(
    f"Notes with any mention      : "
    f"{len(notes_with_any_mention):,}"
)

print(
    f"Notes with affirmed mention : "
    f"{len(notes_with_affirmed_mention):,}"
)

**Category summary**

In [ ]:
# 35. Creating Category Summary

category_summary_df = pd.DataFrame(
    [
        {
            "category": category,
            "mention_count": count
        }
        for category, count
        in category_counter.items()
    ]
)

category_summary_df = (
    category_summary_df
    .sort_values(
        "mention_count",
        ascending=False
    )
    .reset_index(drop=True)
)

category_summary_df[
    "percentage_of_mentions"
] = (
    category_summary_df[
        "mention_count"
    ]
    / total_mentions_scanned
    * 100
)

display(
    category_summary_df
)

**Normalized mention frequency**

In [ ]:
# 36. Creating Normalized Mention Frequency Table

mention_frequency_df = pd.DataFrame(
    normalized_mention_counter.most_common(),
    columns=[
        "normalized_text",
        "mention_count"
    ]
)

mention_frequency_df[
    "percentage_of_mentions"
] = (
    mention_frequency_df[
        "mention_count"
    ]
    / total_mentions_scanned
    * 100
)

print(
    f"Unique normalized mentions: "
    f"{len(mention_frequency_df):,}"
)

display(
    mention_frequency_df.head(30)
)

**Context summary**

In [ ]:
# 37. Creating Context Summary

context_summary_df = pd.DataFrame(
    [
        {
            "context_status": context_name,
            "mention_count": count,
            "percentage_of_mentions": (
                count
                / total_mentions_scanned
                * 100
                if total_mentions_scanned
                else 0
            )
        }
        for context_name, count
        in context_counter.items()
    ]
)

context_summary_df = (
    context_summary_df
    .sort_values(
        "mention_count",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    context_summary_df
)

**Processing statistics**

In [ ]:
# 38. Creating Overall Processing Statistics

total_processing_seconds = (
    manifest_df[
        "processing_seconds"
    ].sum()
)

total_candidate_windows = (
    manifest_df[
        "candidate_windows"
    ].sum()
)

total_original_characters = (
    manifest_df[
        "original_characters"
    ].sum()
)

total_characters_processed = (
    manifest_df[
        "characters_processed"
    ].sum()
)

character_reduction_percentage = (
    100
    * (
        1
        - (
            total_characters_processed
            / total_original_characters
        )
    )
    if total_original_characters
    else 0
)

processing_statistics_df = pd.DataFrame(
    [
        {
            "total_notes": total_notes,

            "total_shards": len(
                production_shards
            ),

            "total_mentions": (
                total_mentions_scanned
            ),

            "notes_with_any_mention": (
                len(
                    notes_with_any_mention
                )
            ),

            "notes_without_any_mention": (
                total_notes
                - len(
                    notes_with_any_mention
                )
            ),

            "notes_with_current_affirmed_mention": (
                len(
                    notes_with_affirmed_mention
                )
            ),

            "unique_categories": (
                len(category_counter)
            ),

            "unique_normalized_mentions": (
                len(
                    normalized_mention_counter
                )
            ),

            "missing_normalized_mentions": (
                missing_normalized_mentions
            ),

            "blank_normalized_mentions": (
                blank_normalized_mentions
            ),

            "candidate_windows": (
                total_candidate_windows
            ),

            "original_characters": (
                total_original_characters
            ),

            "characters_processed": (
                total_characters_processed
            ),

            "character_reduction_percentage": (
                character_reduction_percentage
            ),

            "total_processing_seconds": (
                total_processing_seconds
            ),

            "total_processing_minutes": (
                total_processing_seconds
                / 60
            ),

            "mean_seconds_per_note": (
                total_processing_seconds
                / total_notes
            ),

            "notes_per_second": (
                total_notes
                / total_processing_seconds
                if total_processing_seconds
                else 0
            ),

            "mean_mentions_per_note": (
                total_mentions_scanned
                / total_notes
            )
        }
    ]
)

display(
    processing_statistics_df.T
)

# **Note-level output**

Mention-level extraction is necessary for span evaluation, while note-level output is useful for complication-presence analysis.

**Note-category summary**

In [ ]:
# 39. Creating Note-Level Category Summary

note_category_parts = []

for shard_file in tqdm(
    production_shards,
    desc="Building note-category summary"
):
    shard_df = pd.read_csv(
        shard_file,
        usecols=[
            "note_id",
            "subject_id",
            "hadm_id",
            "category",
            "is_current_affirmed"
        ]
    )

    if shard_df.empty:
        continue

    shard_note_category = (
        shard_df
        .groupby(
            [
                "note_id",
                "subject_id",
                "hadm_id",
                "category"
            ],
            dropna=False
        )
        .agg(
            total_mentions=(
                "category",
                "size"
            ),

            affirmed_mentions=(
                "is_current_affirmed",
                "sum"
            )
        )
        .reset_index()
    )

    shard_note_category[
        "has_any_mention"
    ] = True

    shard_note_category[
        "has_affirmed_mention"
    ] = (
        shard_note_category[
            "affirmed_mentions"
        ] > 0
    )

    note_category_parts.append(
        shard_note_category
    )

note_category_summary_df = pd.concat(
    note_category_parts,
    ignore_index=True
)

note_category_summary_df = (
    note_category_summary_df
    .groupby(
        [
            "note_id",
            "subject_id",
            "hadm_id",
            "category"
        ],
        as_index=False,
        dropna=False
    )
    .agg(
        total_mentions=(
            "total_mentions",
            "sum"
        ),

        affirmed_mentions=(
            "affirmed_mentions",
            "sum"
        ),

        has_any_mention=(
            "has_any_mention",
            "max"
        ),

        has_affirmed_mention=(
            "has_affirmed_mention",
            "max"
        )
    )
)

print(
    f"Note-category rows: "
    f"{len(note_category_summary_df):,}"
)

display(
    note_category_summary_df.head(20)
)

**Creating note-level summary for all notes**

In [ ]:
# 40. Creating Complete Note-Level Summary

all_note_identifiers_df = pd.read_csv(
    INPUT_FILE,
    usecols=[
        "note_id",
        "subject_id",
        "hadm_id"
    ]
)

mention_note_summary_df = (
    note_category_summary_df
    .groupby(
        [
            "note_id",
            "subject_id",
            "hadm_id"
        ],
        as_index=False
    )
    .agg(
        total_complication_mentions=(
            "total_mentions",
            "sum"
        ),

        affirmed_complication_mentions=(
            "affirmed_mentions",
            "sum"
        ),

        unique_complication_categories=(
            "category",
            "nunique"
        ),

        categories_with_affirmed_mentions=(
            "has_affirmed_mention",
            "sum"
        )
    )
)

note_summary_df = (
    all_note_identifiers_df
    .merge(
        mention_note_summary_df,
        on=[
            "note_id",
            "subject_id",
            "hadm_id"
        ],
        how="left",
        validate="one_to_one"
    )
)

numeric_summary_columns = [
    "total_complication_mentions",
    "affirmed_complication_mentions",
    "unique_complication_categories",
    "categories_with_affirmed_mentions"
]

note_summary_df[
    numeric_summary_columns
] = (
    note_summary_df[
        numeric_summary_columns
    ]
    .fillna(0)
    .astype("int64")
)

note_summary_df[
    "has_any_complication_mention"
] = (
    note_summary_df[
        "total_complication_mentions"
    ] > 0
)

note_summary_df[
    "has_current_affirmed_complication"
] = (
    note_summary_df[
        "affirmed_complication_mentions"
    ] > 0
)

assert len(note_summary_df) == total_notes

assert note_summary_df[
    "note_id"
].nunique() == total_notes

print(
    f"Complete note-level rows: "
    f"{len(note_summary_df):,}"
)

display(
    note_summary_df.head()
)

# **Final exports**

**Exporting all summaries**

In [ ]:
# 41. Exporting Final medSpaCy Outputs

CATEGORY_SUMMARY_FILE = (
    SUMMARY_DIRECTORY
    / "medspacy_category_summary.csv"
)

MENTION_FREQUENCY_FILE = (
    SUMMARY_DIRECTORY
    / "medspacy_mention_frequency.csv.gz"
)

CONTEXT_SUMMARY_FILE = (
    SUMMARY_DIRECTORY
    / "medspacy_context_summary.csv"
)

PROCESSING_STATISTICS_FILE = (
    SUMMARY_DIRECTORY
    / "medspacy_processing_statistics.csv"
)

NOTE_CATEGORY_SUMMARY_FILE = (
    SUMMARY_DIRECTORY
    / "medspacy_note_category_summary.csv.gz"
)

NOTE_SUMMARY_FILE = (
    SUMMARY_DIRECTORY
    / "medspacy_note_summary.csv.gz"
)

category_summary_df.to_csv(
    CATEGORY_SUMMARY_FILE,
    index=False
)

mention_frequency_df.to_csv(
    MENTION_FREQUENCY_FILE,
    index=False,
    compression="gzip"
)

context_summary_df.to_csv(
    CONTEXT_SUMMARY_FILE,
    index=False
)

processing_statistics_df.to_csv(
    PROCESSING_STATISTICS_FILE,
    index=False
)

note_category_summary_df.to_csv(
    NOTE_CATEGORY_SUMMARY_FILE,
    index=False,
    compression="gzip"
)

note_summary_df.to_csv(
    NOTE_SUMMARY_FILE,
    index=False,
    compression="gzip"
)

print("-" * 75)
print("FINAL OUTPUT EXPORTS")
print("-" * 75)

for output_file in [
    RULE_FILE,
    MANIFEST_FILE,
    CATEGORY_SUMMARY_FILE,
    MENTION_FREQUENCY_FILE,
    CONTEXT_SUMMARY_FILE,
    PROCESSING_STATISTICS_FILE,
    NOTE_CATEGORY_SUMMARY_FILE,
    NOTE_SUMMARY_FILE
]:
    print(output_file)

**Final accounting checks**

In [ ]:
# 42. Final Accounting and Integrity Checks

assert total_mentions_scanned == int(
    category_summary_df[
        "mention_count"
    ].sum()
)

assert len(note_summary_df) == total_notes

assert (
    note_summary_df[
        "has_any_complication_mention"
    ].sum()
    == len(notes_with_any_mention)
)

assert (
    note_summary_df[
        "has_current_affirmed_complication"
    ].sum()
    == len(
        notes_with_affirmed_mention
    )
)

assert (
    missing_normalized_mentions
    + blank_normalized_mentions
    + sum(
        normalized_mention_counter.values()
    )
    == total_mentions_scanned
)

expected_output_files = [
    RULE_FILE,
    MANIFEST_FILE,
    CATEGORY_SUMMARY_FILE,
    MENTION_FREQUENCY_FILE,
    CONTEXT_SUMMARY_FILE,
    PROCESSING_STATISTICS_FILE,
    NOTE_CATEGORY_SUMMARY_FILE,
    NOTE_SUMMARY_FILE
]

missing_output_files = [
    str(file_path)
    for file_path in expected_output_files
    if not file_path.exists()
]

assert not missing_output_files, (
    f"Missing final output files: "
    f"{missing_output_files}"
)

print("-" * 75)
print("NOTEBOOK 05 COMPLETION STATUS")
print("-" * 75)

print(f"Input notes validated     : {total_notes:,}")
print(f"Production shards         : {len(production_shards):,}")
print(f"Mentions validated        : {total_mentions_scanned:,}")
print(f"Note-level rows           : {len(note_summary_df):,}")
print(f"Missing final outputs     : {len(missing_output_files)}")

print(
    "\n medSpaCy extraction "
    "completed and validated successfully."
)

# **Patient-level analysis**

**Creating patient-category summary**

In [ ]:
# 43. Creating Patient-Level Category Summary

patient_category_summary_df = (
    note_category_summary_df
    .groupby(
        [
            "subject_id",
            "category"
        ],
        as_index=False,
        dropna=False
    )
    .agg(
        admission_count=(
            "hadm_id",
            "nunique"
        ),

        note_count=(
            "note_id",
            "nunique"
        ),

        total_mentions=(
            "total_mentions",
            "sum"
        ),

        affirmed_mentions=(
            "affirmed_mentions",
            "sum"
        ),

        has_any_mention=(
            "has_any_mention",
            "max"
        ),

        has_affirmed_mention=(
            "has_affirmed_mention",
            "max"
        )
    )
)

patient_category_summary_df[
    "has_any_mention"
] = (
    patient_category_summary_df[
        "has_any_mention"
    ]
    .fillna(False)
    .astype(bool)
)

patient_category_summary_df[
    "has_affirmed_mention"
] = (
    patient_category_summary_df[
        "has_affirmed_mention"
    ]
    .fillna(False)
    .astype(bool)
)

print("-" * 75)
print("PATIENT-CATEGORY SUMMARY")
print("-" * 75)

print(
    f"Patient-category rows : "
    f"{len(patient_category_summary_df):,}"
)

print(
    f"Unique patients       : "
    f"{patient_category_summary_df['subject_id'].nunique():,}"
)

print(
    f"Unique categories     : "
    f"{patient_category_summary_df['category'].nunique():,}"
)

display(
    patient_category_summary_df.head(20)
)

**Creating complete patient-level summary**

In [ ]:
# 44. Creating Complete Patient-Level Summary

all_patient_identifiers_df = (
    all_note_identifiers_df[
        [
            "subject_id"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

patient_aggregate_df = (
    patient_category_summary_df
    .groupby(
        "subject_id",
        as_index=False
    )
    .agg(
        total_admissions_with_mentions=(
            "admission_count",
            "sum"
        ),

        total_notes_with_mentions=(
            "note_count",
            "sum"
        ),

        total_complication_mentions=(
            "total_mentions",
            "sum"
        ),

        affirmed_complication_mentions=(
            "affirmed_mentions",
            "sum"
        ),

        unique_complication_categories=(
            "category",
            "nunique"
        ),

        categories_with_affirmed_mentions=(
            "has_affirmed_mention",
            "sum"
        )
    )
)

patient_summary_df = (
    all_patient_identifiers_df
    .merge(
        patient_aggregate_df,
        on="subject_id",
        how="left",
        validate="one_to_one"
    )
)

patient_numeric_columns = [
    "total_admissions_with_mentions",
    "total_notes_with_mentions",
    "total_complication_mentions",
    "affirmed_complication_mentions",
    "unique_complication_categories",
    "categories_with_affirmed_mentions"
]

patient_summary_df[
    patient_numeric_columns
] = (
    patient_summary_df[
        patient_numeric_columns
    ]
    .fillna(0)
    .astype("int64")
)

patient_summary_df[
    "has_any_complication_mention"
] = (
    patient_summary_df[
        "total_complication_mentions"
    ] > 0
)

patient_summary_df[
    "has_current_affirmed_complication"
] = (
    patient_summary_df[
        "affirmed_complication_mentions"
    ] > 0
)

expected_patient_count = (
    all_note_identifiers_df[
        "subject_id"
    ].nunique()
)

assert len(patient_summary_df) == expected_patient_count

assert (
    patient_summary_df[
        "subject_id"
    ].nunique()
    == expected_patient_count
)

print("-" * 75)
print("COMPLETE PATIENT SUMMARY")
print("-" * 75)

print(
    f"Patient-level rows       : "
    f"{len(patient_summary_df):,}"
)

print(
    f"Patients with mentions   : "
    f"{patient_summary_df['has_any_complication_mention'].sum():,}"
)

print(
    f"Patients with affirmed   : "
    f"{patient_summary_df['has_current_affirmed_complication'].sum():,}"
)

display(
    patient_summary_df.head()
)

**Creating patient-category binary matrix**

This produces one row per patient and one binary column for each complication category.

In [ ]:
# 45. Creating Patient-Category Binary Matrix

affirmed_patient_category_df = (
    patient_category_summary_df[
        patient_category_summary_df[
            "has_affirmed_mention"
        ]
    ][
        [
            "subject_id",
            "category"
        ]
    ]
    .drop_duplicates()
)

patient_category_matrix_df = (
    affirmed_patient_category_df
    .assign(present=1)
    .pivot_table(
        index="subject_id",
        columns="category",
        values="present",
        aggfunc="max",
        fill_value=0
    )
    .reset_index()
)

patient_category_matrix_df.columns.name = None

category_matrix_columns = [
    column
    for column in patient_category_matrix_df.columns
    if column != "subject_id"
]

patient_category_matrix_df[
    category_matrix_columns
] = (
    patient_category_matrix_df[
        category_matrix_columns
    ]
    .astype("int8")
)

patient_category_matrix_df = (
    all_patient_identifiers_df
    .merge(
        patient_category_matrix_df,
        on="subject_id",
        how="left",
        validate="one_to_one"
    )
)

patient_category_matrix_df[
    category_matrix_columns
] = (
    patient_category_matrix_df[
        category_matrix_columns
    ]
    .fillna(0)
    .astype("int8")
)

patient_category_matrix_df[
    "number_of_affirmed_categories"
] = (
    patient_category_matrix_df[
        category_matrix_columns
    ]
    .sum(axis=1)
    .astype("int16")
)

assert len(patient_category_matrix_df) == expected_patient_count

print("-" * 75)
print("PATIENT-CATEGORY BINARY MATRIX")
print("-" * 75)

print(
    f"Patients           : "
    f"{len(patient_category_matrix_df):,}"
)

print(
    f"Category columns   : "
    f"{len(category_matrix_columns):,}"
)

display(
    patient_category_matrix_df.head()
)

# **Category prevalence**

**Calculating note-level and patient-level prevalence**

In [ ]:
# 46. Calculating Category Prevalence

total_patient_count = (
    patient_summary_df[
        "subject_id"
    ].nunique()
)

category_prevalence_records = []

for category in sorted(
    note_category_summary_df[
        "category"
    ].dropna().unique()
):
    category_note_data = (
        note_category_summary_df[
            note_category_summary_df[
                "category"
            ] == category
        ]
    )

    category_patient_data = (
        patient_category_summary_df[
            patient_category_summary_df[
                "category"
            ] == category
        ]
    )

    notes_with_any = int(
        category_note_data[
            "note_id"
        ].nunique()
    )

    notes_with_affirmed = int(
        category_note_data.loc[
            category_note_data[
                "has_affirmed_mention"
            ],
            "note_id"
        ].nunique()
    )

    patients_with_any = int(
        category_patient_data[
            "subject_id"
        ].nunique()
    )

    patients_with_affirmed = int(
        category_patient_data.loc[
            category_patient_data[
                "has_affirmed_mention"
            ],
            "subject_id"
        ].nunique()
    )

    category_prevalence_records.append(
        {
            "category": category,

            "notes_with_any_mention": (
                notes_with_any
            ),

            "note_prevalence_any_percentage": (
                notes_with_any
                / total_notes
                * 100
            ),

            "notes_with_affirmed_mention": (
                notes_with_affirmed
            ),

            "note_prevalence_affirmed_percentage": (
                notes_with_affirmed
                / total_notes
                * 100
            ),

            "patients_with_any_mention": (
                patients_with_any
            ),

            "patient_prevalence_any_percentage": (
                patients_with_any
                / total_patient_count
                * 100
            ),

            "patients_with_affirmed_mention": (
                patients_with_affirmed
            ),

            "patient_prevalence_affirmed_percentage": (
                patients_with_affirmed
                / total_patient_count
                * 100
            )
        }
    )

category_prevalence_df = pd.DataFrame(
    category_prevalence_records
)

category_prevalence_df = (
    category_prevalence_df
    .sort_values(
        [
            "notes_with_affirmed_mention",
            "notes_with_any_mention"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

print("-" * 75)
print("CATEGORY PREVALENCE")
print("-" * 75)

display(
    category_prevalence_df
)

# **Quality-assurance sample**

**Creating a stratified manual-review sample**

In [ ]:
# 47. Creating Stratified Quality-Assurance Sample

QA_SAMPLE_PER_STATUS = 20
QA_RANDOM_SEED = 42

qa_status_definitions = {
    "current_affirmed": (
        lambda df: df[
            "is_current_affirmed"
        ]
    ),

    "negated": (
        lambda df: df[
            "is_negated"
        ]
    ),

    "uncertain": (
        lambda df: df[
            "is_uncertain"
        ]
    ),

    "historical": (
        lambda df: df[
            "is_historical"
        ]
    ),

    "hypothetical": (
        lambda df: df[
            "is_hypothetical"
        ]
    ),

    "family": (
        lambda df: df[
            "is_family"
        ]
    )
}

qa_sample_parts = []

for status_name, status_filter in (
    qa_status_definitions.items()
):
    eligible_parts = []

    for shard_file in production_shards:
        shard_sample_source = pd.read_csv(
            shard_file,
            usecols=[
                "note_id",
                "subject_id",
                "hadm_id",
                "entity_text",
                "normalized_text",
                "category",
                "sentence_text",
                "is_negated",
                "is_uncertain",
                "is_historical",
                "is_hypothetical",
                "is_family",
                "is_current_affirmed"
            ]
        )

        if shard_sample_source.empty:
            continue

        boolean_mask = status_filter(
            shard_sample_source
        ).fillna(False).astype(bool)

        eligible_shard_rows = (
            shard_sample_source[
                boolean_mask
            ]
        )

        if not eligible_shard_rows.empty:
            eligible_parts.append(
                eligible_shard_rows
            )

    if not eligible_parts:
        continue

    eligible_status_df = pd.concat(
        eligible_parts,
        ignore_index=True
    )

    sample_size = min(
        QA_SAMPLE_PER_STATUS,
        len(eligible_status_df)
    )

    sampled_status_df = (
        eligible_status_df
        .sample(
            n=sample_size,
            random_state=QA_RANDOM_SEED
        )
        .copy()
    )

    sampled_status_df[
        "sampling_status"
    ] = status_name

    qa_sample_parts.append(
        sampled_status_df
    )

qa_sample_df = pd.concat(
    qa_sample_parts,
    ignore_index=True
)

qa_sample_df.insert(
    0,
    "qa_record_id",
    [
        f"QA_{index:04d}"
        for index in range(
            1,
            len(qa_sample_df) + 1
        )
    ]
)

qa_sample_df[
    "manual_correct_entity"
] = ""

qa_sample_df[
    "manual_correct_category"
] = ""

qa_sample_df[
    "manual_correct_context"
] = ""

qa_sample_df[
    "reviewer_comment"
] = ""

print("-" * 75)
print("QUALITY-ASSURANCE SAMPLE")
print("-" * 75)

print(
    f"QA records created: "
    f"{len(qa_sample_df):,}"
)

display(
    qa_sample_df[
        [
            "qa_record_id",
            "sampling_status",
            "entity_text",
            "category",
            "sentence_text"
        ]
    ].head(20)
)

# **Export remaining outputs**

**Exporting patient and QA outputs**

In [ ]:
# 48. Exporting Patient-Level and QA Outputs

PATIENT_CATEGORY_SUMMARY_FILE = (
    SUMMARY_DIRECTORY
    / "medspacy_patient_category_summary.csv.gz"
)

PATIENT_SUMMARY_FILE = (
    SUMMARY_DIRECTORY
    / "medspacy_patient_summary.csv.gz"
)

PATIENT_CATEGORY_MATRIX_FILE = (
    SUMMARY_DIRECTORY
    / "medspacy_patient_category_matrix.csv.gz"
)

CATEGORY_PREVALENCE_FILE = (
    SUMMARY_DIRECTORY
    / "medspacy_category_prevalence.csv"
)

QA_SAMPLE_FILE = (
    SUMMARY_DIRECTORY
    / "medspacy_quality_assurance_sample.csv"
)

patient_category_summary_df.to_csv(
    PATIENT_CATEGORY_SUMMARY_FILE,
    index=False,
    compression="gzip"
)

patient_summary_df.to_csv(
    PATIENT_SUMMARY_FILE,
    index=False,
    compression="gzip"
)

patient_category_matrix_df.to_csv(
    PATIENT_CATEGORY_MATRIX_FILE,
    index=False,
    compression="gzip"
)

category_prevalence_df.to_csv(
    CATEGORY_PREVALENCE_FILE,
    index=False
)

qa_sample_df.to_csv(
    QA_SAMPLE_FILE,
    index=False
)

print("-" * 75)
print("ADDITIONAL OUTPUT EXPORTS")
print("-" * 75)

for output_file in [
    PATIENT_CATEGORY_SUMMARY_FILE,
    PATIENT_SUMMARY_FILE,
    PATIENT_CATEGORY_MATRIX_FILE,
    CATEGORY_PREVALENCE_FILE,
    QA_SAMPLE_FILE
]:
    print(output_file)

# **Final pipeline configuration record**

**Saving reproducibility configuration**

In [ ]:
# 49. Saving Reproducibility Configuration

pipeline_configuration = {
    "notebook": (
        "05_medspacy_clinical_complication_extraction"
    ),

    "input_file": str(
        INPUT_FILE
    ),

    "input_note_count": int(
        total_notes
    ),

    "python_version": (
        sys.version.split()[0]
    ),

    "pandas_version": (
        pd.__version__
    ),

    "spacy_version": (
        spacy.__version__
    ),

    "medspacy_version": (
        medspacy.__version__
    ),

    "pipeline_components": (
        medspacy_nlp.pipe_names
    ),

    "number_of_complication_categories": int(
        number_of_categories
    ),

    "number_of_target_rules": int(
        len(complication_rules_df)
    ),

    "candidate_prefilter_enabled": True,

    "maximum_segment_characters": 1500,

    "neighbour_segments_included": True,

    "production_chunk_size": int(
        FULL_RUN_CHUNK_SIZE
    ),

    "production_shard_count": int(
        len(production_shards)
    ),

    "production_mention_count": int(
        total_mentions_scanned
    ),

    "random_seed": int(
        RANDOM_SEED
    ),

    "custom_uncertainty_rules": True,

    "custom_hypothetical_rules": True
}

PIPELINE_CONFIGURATION_FILE = (
    SUMMARY_DIRECTORY
    / "medspacy_pipeline_configuration.json"
)

with open(
    PIPELINE_CONFIGURATION_FILE,
    "w",
    encoding="utf-8"
) as configuration_file:
    json.dump(
        pipeline_configuration,
        configuration_file,
        indent=4
    )

print("-" * 75)
print("PIPELINE CONFIGURATION")
print("-" * 75)

print(
    json.dumps(
        pipeline_configuration,
        indent=4
    )
)

print(
    f"\nSaved to: "
    f"{PIPELINE_CONFIGURATION_FILE}"
)

# **Comprehensive final validation**

**Validating all final  outputs**

In [ ]:
# 50. Comprehensive Final Validation

final_notebook_05_files = [
    RULE_FILE,
    MANIFEST_FILE,
    CATEGORY_SUMMARY_FILE,
    MENTION_FREQUENCY_FILE,
    CONTEXT_SUMMARY_FILE,
    PROCESSING_STATISTICS_FILE,
    NOTE_CATEGORY_SUMMARY_FILE,
    NOTE_SUMMARY_FILE,
    PATIENT_CATEGORY_SUMMARY_FILE,
    PATIENT_SUMMARY_FILE,
    PATIENT_CATEGORY_MATRIX_FILE,
    CATEGORY_PREVALENCE_FILE,
    QA_SAMPLE_FILE,
    PIPELINE_CONFIGURATION_FILE
]

missing_final_files = [
    str(file_path)
    for file_path in final_notebook_05_files
    if not file_path.exists()
]

empty_final_files = [
    str(file_path)
    for file_path in final_notebook_05_files
    if (
        file_path.exists()
        and file_path.stat().st_size == 0
    )
]

assert not missing_final_files, (
    f"Missing final files: "
    f"{missing_final_files}"
)

assert not empty_final_files, (
    f"Empty final files: "
    f"{empty_final_files}"
)

assert len(note_summary_df) == (
    total_notes
)

assert len(patient_summary_df) == (
    expected_patient_count
)

assert len(patient_category_matrix_df) == (
    expected_patient_count
)

assert (
    patient_summary_df[
        "total_complication_mentions"
    ].sum()
    == total_mentions_scanned
)

assert (
    patient_category_summary_df[
        "total_mentions"
    ].sum()
    == total_mentions_scanned
)

assert set(
    category_prevalence_df[
        "category"
    ]
) == set(
    category_summary_df[
        "category"
    ]
)

assert (
    patient_category_matrix_df[
        "number_of_affirmed_categories"
    ].ge(0).all()
)

print("-" * 75)
print("FINAL VALIDATION")
print("-" * 75)

print(
    f"Input notes                    : "
    f"{total_notes:,}"
)

print(
    f"Unique patients                : "
    f"{expected_patient_count:,}"
)

print(
    f"Production shards              : "
    f"{len(production_shards):,}"
)

print(
    f"Validated mentions             : "
    f"{total_mentions_scanned:,}"
)

print(
    f"Note-level rows                : "
    f"{len(note_summary_df):,}"
)

print(
    f"Patient-level rows             : "
    f"{len(patient_summary_df):,}"
)

print(
    f"Patient-category matrix rows   : "
    f"{len(patient_category_matrix_df):,}"
)

print(
    f"QA review records              : "
    f"{len(qa_sample_df):,}"
)

print(
    f"Final output files             : "
    f"{len(final_notebook_05_files):,}"
)

print(
    f"Missing files                  : "
    f"{len(missing_final_files)}"
)

print(
    f"Empty files                    : "
    f"{len(empty_final_files)}"
)


# **Conclusion**

# Conclusion

A lightweight medSpaCy clinical NLP pipeline was developed to identify
predefined clinical complications in ICU discharge summaries.

The pipeline used:

- a blank English spaCy tokenizer;
- spaCy rule-based sentence segmentation;
- the medSpaCy target matcher;
- medSpaCy ConText;
- a curated clinical complication terminology;
- task-specific uncertainty and hypothetical-context rules;
- candidate-window prefiltering;
- resumable shard-based processing.

The complete cohort contained 65,323 ICU discharge summaries. All notes were
processed successfully across 262 production shards, producing 379,142
mention-level complication records.

The extraction outputs were validated for:

- complete note coverage;
- continuous shard coverage;
- consistent output schemas;
- agreement between shard and manifest mention counts;
- complete note-level aggregation;
- complete patient-level aggregation;
- successful generation of all required output files.

The notebook produced:

1. mention-level complication extractions;
2. contextual attributes for each mention;
3. note-category summaries;
4. complete note-level summaries;
5. patient-category summaries;
6. complete patient-level summaries;
7. a patient-by-category binary matrix;
8. category prevalence statistics;
9. processing and runtime statistics;
10. a stratified quality-assurance sample;
11. a reproducibility configuration record.

The resulting medSpaCy outputs are ready for manual review and subsequent
comparison with the scispaCy pipeline using precision, recall and F1-score.